# Fine-Tuning TinyLlama for Sentiment Classification

This chapter documents the fine-tuning setup for TinyLlama on the sentiment task. Code execution is disabled in Quarto because training will be run in Colab, not during book rendering.

## Install dependencies

This block installs the libraries needed for QLoRA fine-tuning in Colab. We use `transformers` for the base model, `peft` for LoRA adapters, `trl` for supervised fine-tuning, and `bitsandbytes` for 4-bit quantization.

In [3]:
#| label: install-dependencies
#| echo: true

!pip install -q transformers peft trl bitsandbytes accelerate datasets pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.0 MB/s eta 0:00:00


## Import packages

This block imports the core libraries for loading data, configuring the model, fine-tuning with QLoRA, and evaluating the predictions.

In [4]:
#| label: imports
#| echo: true

import pandas as pd
import numpy as np
import torch

from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig
from trl import SFTTrainer

from sklearn.metrics import f1_score, classification_report

## Local paths for Positron / Quarto

This block resolves the repository root and constructs local file paths. It is useful when writing and documenting the notebook in Positron before converting it to a Colab notebook.

In [5]:
#| label: paths-and-config
#| echo: true

# _cwd = Path.cwd()
# REPO_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parents[1]
# STAGE2_DIR = REPO_ROOT / "data/processed/stage2"

SEED = 42

# train_path = STAGE2_DIR / "llm_train.parquet"
# test_path = STAGE2_DIR / "llm_test.parquet"

For usage in **Positron/Quarto**, use the next block. For **Google Colab**, use the Colab block further below instead.

## Load data locally

This block loads the train and test files from the local repository structure. It is only used in the local Quarto / Positron workflow.

In [6]:
#| label: load-data
#| echo: true

# train_df = pd.read_parquet(train_path)
# test_df = pd.read_parquet(test_path)

# train_df.shape, test_df.shape

## Alternative: load data in Colab

This alternative block mounts Google Drive and loads the train and test parquet files from the shared Colab folder. Use this block in Colab instead of the local path/loading blocks above.

In [7]:
#| label: colab-drive-and-load-data
#| echo: true

from google.colab import drive

drive.mount('/content/drive')

train_path = "/content/drive/MyDrive/CLT/stage2/llm_train.parquet"
test_path = "/content/drive/MyDrive/CLT/stage2/llm_test.parquet"

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

train_df.shape, test_df.shape

KeyboardInterrupt: 

In [ ]:
train_df["sentiment_label"] = train_df["sentiment_label"].str.replace("(partly) negative", "negative")
test_df["sentiment_label"] = test_df["sentiment_label"].str.replace("(partly) negative", "negative")

## Inspect the data

This block shows a few examples of the prompt text and the target sentiment label. It verifies that the dataset is already in instruction format and ready for supervised fine-tuning.

In [ ]:
#| label: inspect-data
#| echo: true

train_df[["text", "sentiment_label"]].head(3)

## Define labels and model name

This block defines the label set, the input and target columns, and the selected base model. Keeping these settings in one place makes the notebook easier to adjust later.

In [ ]:
#| label: define-labels
#| echo: true

LABELS = ["negative", "neutral", "positive"]
TARGET_COLUMN = "sentiment_label"
TEXT_COLUMN = "text"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

## Configure 4-bit quantization

This block sets up the quantization configuration for QLoRA. We use 4-bit loading to reduce GPU memory usage, which is important for running TinyLlama efficiently on Colab Pro.

In [ ]:
#| label: quantization-config
#| echo: true

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

## Load tokenizer and model

This block loads the tokenizer and the quantized base model. If the tokenizer has no padding token, we reuse the EOS token so batching works correctly.

In [ ]:
#| label: load-tokenizer-and-model
#| echo: true

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

## Configure LoRA adapters

This block defines the LoRA setup. The chosen values are conservative starter settings: small enough for a first Colab test, but still standard for parameter-efficient fine-tuning.

In [ ]:
#| label: lora-config
#| echo: true

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## Prepare dataset

This block prepares the full training and test datasets for fine-tuning and evaluation. The `text` column contains the Alpaca-formatted instruction, the `sentiment_label` column contains the gold label used for evaluation after inference.

In [ ]:
#| label: prepare-datasets
#| echo: true

train_dataset = Dataset.from_pandas(train_df[[TEXT_COLUMN]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[[TEXT_COLUMN, TARGET_COLUMN]], preserve_index=False)

## Define training arguments

This block sets small training parameters for the initial test run. The batch size is intentionally small to fit Colab GPU memory, and we train only for one epoch to validate the pipeline quickly.

In [ ]:
#| label: training-args
#| echo: true

training_args = TrainingArguments(
    output_dir="outputs/tinyllama-sentiment",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    bf16=True,
    fp16=False,
)

## Create the supervised fine-tuning trainer

This block combines the base model, LoRA configuration, training dataset, and training arguments into one trainer object. The trainer uses the `text` column as the input field.

In [ ]:
#| label: sft-trainer
#| echo: true

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)

## Run the first training pass

This block starts fine-tuning. In the first Colab test, it should only confirm that the pipeline runs successfully on the small debug sample.

In [ ]:
#| label: run-training
#| echo: true

trainer.train()

In [ ]:
#| label: save-model
#| echo: true

trainer.model.save_pretrained("/content/drive/MyDrive/CLT/stage2/tinyllama-sentiment")
tokenizer.save_pretrained("/content/drive/MyDrive/CLT/stage2/tinyllama-sentiment")
print("Model saved to Google Drive.")


## Generate predictions on the debug test set

This block applies the fine-tuned model to a few held-out examples. We use only a small number of prompts and keep the generation short because the target output is just one sentiment label.

In [ ]:
#| label: make-inference-prompt
#| echo: true

def make_inference_prompt(full_text: str) -> str:
    parts = full_text.split("### Response:")
    instruction = parts[0]
    words = instruction.split()
    if len(words) > 500:
        instruction = " ".join(words[:500]) + "\n\n"
    return instruction + "### Response:"

In [ ]:
#| label: generate-predictions
#| echo: true

generator = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_new_tokens=12,
    return_full_text=False,
)

sample_prompts = [make_inference_prompt(t) for t in test_df[TEXT_COLUMN].tolist()]
sample_outputs = generator(sample_prompts)

sample_outputs

## Extract labels from model outputs

This block maps the generated text back to one of the three sentiment labels. Because we set `return_full_text=False`, the extraction runs only on the model output, not on the original prompt.

In [ ]:
#| label: extract-labels
#| echo: true

def extract_label(generated_text: str) -> str:
    text = generated_text.strip().lower()
    if "negative" in text:
        return "negative"
    elif "neutral" in text:
        return "neutral"
    elif "positive" in text:
        return "positive"
    else:
        return "unknown"

pred_labels = []
true_labels = test_df[TARGET_COLUMN].tolist()

for output in sample_outputs:
    generated_text = output[0]["generated_text"]
    pred_labels.append(extract_label(generated_text))

list(zip(true_labels, pred_labels))

## Compute Macro-F1

This block computes Macro-F1 on the predictions that could be mapped to one of the valid labels. Macro-F1 is the main evaluation metric because it gives equal weight to all sentiment classes.

In [ ]:
#| label: evaluate-macro-f1
#| echo: true

valid_idx = [i for i, pred in enumerate(pred_labels) if pred in LABELS]

y_true = [true_labels[i] for i in valid_idx]
y_pred = [pred_labels[i] for i in valid_idx]

macro_f1 = f1_score(y_true, y_pred, labels=LABELS, average="macro")

print(f"Valid predictions: {len(valid_idx)} / {len(pred_labels)}")
print("Macro-F1:", macro_f1)
print(classification_report(y_true, y_pred, labels=LABELS))

In [ ]:
#| label: save-results
#| echo: true

import os

results_df = pd.DataFrame({"true": y_true, "pred": y_pred})
results_df.to_csv("/content/drive/MyDrive/CLT/stage2/tinyllama_predictions.csv", index=False)

with open("/content/drive/MyDrive/CLT/stage2/tinyllama_results.txt", "w") as f:
    f.write(f"Valid predictions: {len(valid_idx)} / {len(pred_labels)}\n")
    f.write(f"Macro-F1: {macro_f1}\n")
    f.write(classification_report(y_true, y_pred, labels=LABELS))

print("Results saved to Google Drive.")


## Notes

For the first Colab test, this notebook should be run on the small debug sample. After the pipeline works end to end, the same setup can be scaled to the full training and test data.